# Лекция 14. Прогноз цены ETLN: Gradient Boosting vs LSTM


## 1. Установка библиотек

Если запускаете в чистом окружении, выполните ячейку ниже один раз.


In [ ]:
# Установка пакетов (Colab)
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Colab detected -> installing packages...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib', 'scikit-learn'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow==2.16.2'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cachetools', 'deprecation', 'protobuf<5', 'grpcio>=1.59.3', 'python-dateutil'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'git+https://github.com/RussianInvestments/invest-python.git@0.2.0-beta97'])
    print('If this is the first install run: Runtime -> Restart runtime')
else:
    print('Local run detected: use your venv (no install in notebook)')



## 2. Импорты и параметры

Ниже логика подключения к T-Invest сохранена в той же форме, что и в примере (через `Client(...)` и `get_all_candles(...)`).


In [104]:
import os
import sys
from datetime import timedelta
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf

from tinkoff.invest import Client, CandleInterval
from tinkoff.invest.utils import now
from tinkoff.invest.schemas import InstrumentIdType

IN_COLAB = 'google.colab' in sys.modules

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

# Токен: сначала пробуем из переменной окружения, иначе запрашиваем скрытым вводом
TOKEN = os.environ.get('TINVEST_TOKEN', '').strip()
if not TOKEN:
    TOKEN = getpass('Enter T-Invest token (hidden input): ').strip()
    os.environ['TINVEST_TOKEN'] = TOKEN

APP_NAME = 'lecture13-etln-colab'
FIGI = 'BBG00RM6M4T8'  # ETLN (Etalon Group, MOEX)
DAYS_BACK = 1460
WINDOW = 20             # lookback window in trading days
FORECAST_HORIZON = 3    # direct forecast for 3 trading days ahead
RANDOM_SEED = 42

OUTPUT_DIR = Path('reports/simple_lstm_etln_h3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR.resolve() if not IN_COLAB else OUTPUT_DIR)


def q_to_float(q):
    return float(q.units + q.nano / 1e9)



## 3. Загрузка дневных свечей Эталон Груп (MOEX)

Ниже та же структура подключения через `Client(...)` и `get_all_candles(...)`.

Важно: для **реального токена** используем production endpoint (без `target=INVEST_GRPC_API_SANDBOX`).


In [105]:
rows = []

with Client(TOKEN, app_name=APP_NAME) as api:
    for candle in api.get_all_candles(
        figi=FIGI,
        from_=now() - timedelta(days=DAYS_BACK),
        to=now(),
        interval=CandleInterval.CANDLE_INTERVAL_DAY,
    ):
        rows.append({'Date': candle.time, 'Close': q_to_float(candle.close)})

df = pd.DataFrame(rows)
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df = df.drop_duplicates('Date').sort_values('Date').set_index('Date')
close = df['Close']

close.tail(50)


Date
2025-12-27 00:00:00+00:00    72.950
2025-12-28 00:00:00+00:00    72.920
2025-12-29 00:00:00+00:00    72.170
2025-12-30 00:00:00+00:00    72.220
2026-01-05 00:00:00+00:00    71.550
2026-01-06 00:00:00+00:00    71.500
2026-01-08 00:00:00+00:00    71.070
2026-01-09 00:00:00+00:00    71.220
2026-01-12 00:00:00+00:00    71.340
2026-01-13 00:00:00+00:00    71.160
2026-01-14 00:00:00+00:00    70.910
2026-01-15 00:00:00+00:00    71.130
2026-01-16 00:00:00+00:00    71.980
2026-01-17 00:00:00+00:00    72.050
2026-01-18 00:00:00+00:00    72.090
2026-01-19 00:00:00+00:00    71.910
2026-01-20 00:00:00+00:00    71.850
2026-01-21 00:00:00+00:00    72.720
2026-01-22 00:00:00+00:00    72.820
2026-01-23 00:00:00+00:00    73.120
2026-01-24 00:00:00+00:00    72.940
2026-01-25 00:00:00+00:00    73.140
2026-01-26 00:00:00+00:00    72.290
2026-01-27 00:00:00+00:00    72.920
2026-01-28 00:00:00+00:00    73.280
2026-01-29 00:00:00+00:00    77.750
2026-01-30 00:00:00+00:00    77.470
2026-01-31 00:00:00+00:

## 4. Быстрый просмотр данных


In [ ]:
df = df.copy()
df.index = df.index.tz_convert(None)

print(f'Количество наблюдений: {len(df)}')
print(f'Период: {df.index.min().date()} -> {df.index.max().date()}')
print('Пропуски в Close:', int(df['Close'].isna().sum()))

df.head()


In [ ]:
ax = df['Close'].plot(title='Цена закрытия ETLN (дневные свечи, MOEX)')
ax.set_ylabel('RUB')
plt.show()


## 5. Подготовка целевого ряда (обычная цена `Close`)

Почему тут есть `n * 0.70` и `n * 0.85`:
- `n` — общее число наблюдений в ряду;
- `n * 0.70` — берем первые **70%** точек в `train`;
- `n * 0.85` — это граница **70% + 15%** (то есть конец `valid`).

Модель будет предсказывать цену **через `FORECAST_HORIZON` торговых дней**.


In [6]:
data = df[['Close']].copy()

n = len(data)

TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15  # для читаемости (70% + 15% + 15% = 100%)

train_size = int(n * TRAIN_RATIO)   # 70% наблюдений
valid_size = int(n * VALID_RATIO)   # 15% наблюдений
valid_end = train_size + valid_size # 85% от ряда = train + valid

train_boundary = data.index[train_size]
valid_boundary = data.index[valid_end]

print('Размеры выборок (по точкам ряда до окон):')
print('train:', train_size)
print('valid:', valid_size)
print('test :', n - valid_end)

data.tail()


NameError: name 'df' is not defined

In [ ]:
ax = data['Close'].plot(title='Ряд для модели: Close (цена закрытия ETLN)')
ax.axvline(train_boundary, color='tab:orange', linestyle='--', label='граница train/valid')
ax.axvline(valid_boundary, color='tab:red', linestyle='--', label='граница valid/test')
ax.legend()
ax.set_ylabel('RUB')
plt.show()


## 6. Формируем окна для моделей

**Окно** — это последние `WINDOW` значений цены, по которым модель делает прогноз.

Пример:
- вход: последние `WINDOW` дней
- цель: цена через `FORECAST_HORIZON` торговых дней

Зачем это нужно:
- так мы превращаем временной ряд в обычную задачу обучения с учителем (`X -> y`);
- `Gradient Boosting` и `LSTM` получают данные в понятном формате.


In [ ]:
values = data['Close'].astype('float32').values
dates = data.index

X_all, y_all, idx_all = [], [], []
# target = цена через FORECAST_HORIZON дней после последней точки окна
for i in range(WINDOW, len(values) - FORECAST_HORIZON + 1):
    X_all.append(values[i - WINDOW:i])
    target_pos = i + FORECAST_HORIZON - 1
    y_all.append(values[target_pos])
    idx_all.append(dates[target_pos])

X_all = np.asarray(X_all, dtype='float32')
y_all = np.asarray(y_all, dtype='float32')
idx_all = pd.DatetimeIndex(idx_all)

train_mask = idx_all < train_boundary
valid_mask = (idx_all >= train_boundary) & (idx_all < valid_boundary)
test_mask = idx_all >= valid_boundary

X_train = X_all[train_mask]
y_train = y_all[train_mask]
idx_train = idx_all[train_mask]

X_valid = X_all[valid_mask]
y_valid = y_all[valid_mask]
idx_valid = idx_all[valid_mask]

X_test = X_all[test_mask]
y_test = y_all[test_mask]
idx_test = idx_all[test_mask]

if min(len(X_train), len(X_valid), len(X_test)) == 0:
    raise ValueError('Слишком мало данных для выбранного WINDOW / FORECAST_HORIZON и разбиения.')

# Честное масштабирование: считаем параметры только по train
x_mean = X_train.mean()
x_std = X_train.std() + 1e-8

y_mean = y_train.mean()
y_std = y_train.std() + 1e-8

X_train_scaled = (X_train - x_mean) / x_std
X_valid_scaled = (X_valid - x_mean) / x_std
X_test_scaled = (X_test - x_mean) / x_std

y_train_scaled = (y_train - y_mean) / y_std
y_valid_scaled = (y_valid - y_mean) / y_std
y_test_scaled = (y_test - y_mean) / y_std

# Gradient Boosting принимает 2D-массив [samples, features]
X_train_gb, X_valid_gb, X_test_gb = X_train_scaled, X_valid_scaled, X_test_scaled

# LSTM принимает 3D-массив [samples, timesteps, features]
X_train_lstm = X_train_scaled[..., None]
X_valid_lstm = X_valid_scaled[..., None]
X_test_lstm = X_test_scaled[..., None]

print('Пример одного окна:')
print('X_train_gb[0].shape =', X_train_gb[0].shape)
print('y_train[0] (RUB) =', float(y_train[0]))
print('y_train_scaled[0] =', float(y_train_scaled[0]))
print('Дата целевого значения для первого окна =', idx_train[0].date())

print()
print('Параметры масштабирования (только train):')
print('x_mean =', float(x_mean), 'x_std =', float(x_std))
print('y_mean =', float(y_mean), 'y_std =', float(y_std))

print()
print('Формы массивов:')
print('X_train_gb  :', X_train_gb.shape)
print('X_train_lstm:', X_train_lstm.shape)
print('X_valid_lstm:', X_valid_lstm.shape)
print('X_test_lstm :', X_test_lstm.shape)


## 7. Базовая модель: Gradient Boosting

Ниже обе модели обучаются на одинаково масштабированных данных (параметры масштабирования посчитаны только по `train`).


In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    random_state=RANDOM_SEED,
)

gb_model.fit(X_train_gb, y_train_scaled)

gb_valid_pred_scaled = gb_model.predict(X_valid_gb)
gb_test_pred_scaled = gb_model.predict(X_test_gb)

# Возвращаем прогнозы в рубли

gb_valid_pred = gb_valid_pred_scaled * y_std + y_mean
gb_test_pred = gb_test_pred_scaled * y_std + y_mean

print('Gradient Boosting обучен.')


## 8. LSTM-модель

LSTM обучаем на тех же масштабированных данных, что и Gradient Boosting. После прогноза вернем значения обратно в RUB.


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, 1)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1),
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae'],
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
)

history = lstm_model.fit(
    X_train_lstm,
    y_train_scaled,
    validation_data=(X_valid_lstm, y_valid_scaled),
    epochs=80,
    batch_size=32,
    verbose=0,
    callbacks=[early_stop],
    shuffle=False,
)

pd.DataFrame(history.history).plot(title='Обучение LSTM (на масштабированных данных)')
plt.show()

print('LSTM обучена. Лучшая эпоха:', int(np.argmin(history.history['val_loss'])) + 1)


## 9. Сравнение моделей по цене (`Close`, RUB)

Здесь метрики считаются для прогноза **через `FORECAST_HORIZON` торговых дней**.


In [ ]:
lstm_valid_pred_scaled = lstm_model.predict(X_valid_lstm, verbose=0).ravel()
lstm_test_pred_scaled = lstm_model.predict(X_test_lstm, verbose=0).ravel()

# Возвращаем прогнозы LSTM в рубли
lstm_valid_pred = lstm_valid_pred_scaled * y_std + y_mean
lstm_test_pred = lstm_test_pred_scaled * y_std + y_mean

comparison = pd.DataFrame([
    {
        'model': 'Gradient Boosting',
        'split': 'valid',
        'MAE': mean_absolute_error(y_valid, gb_valid_pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, gb_valid_pred)),
    },
    {
        'model': 'LSTM',
        'split': 'valid',
        'MAE': mean_absolute_error(y_valid, lstm_valid_pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, lstm_valid_pred)),
    },
    {
        'model': 'Gradient Boosting',
        'split': 'test',
        'MAE': mean_absolute_error(y_test, gb_test_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, gb_test_pred)),
    },
    {
        'model': 'LSTM',
        'split': 'test',
        'MAE': mean_absolute_error(y_test, lstm_test_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, lstm_test_pred)),
    },
]).sort_values(['split', 'RMSE'])

comparison


In [ ]:
valid_plot = pd.DataFrame({
    'fact': y_valid,
    'gb': gb_valid_pred,
    'lstm': lstm_valid_pred,
}, index=idx_valid)

ax = valid_plot.tail(120).plot(title=f'Валидация: цена ETLN (цель через {FORECAST_HORIZON} торговых дней)')
ax.set_ylabel('RUB')
plt.show()


## 10. Графики прогноза (цена в RUB)

1. Сравнение на тестовой выборке (цель: цена через `FORECAST_HORIZON` дней).
2. Наглядный график: текущая цена (синяя линия) и справа отрезки до прогноза `GB` и `LSTM` на горизонт `FORECAST_HORIZON`.


In [ ]:
test_price_plot = pd.DataFrame({
    'fact_price': y_test,
    'gb_price': gb_test_pred,
    'lstm_price': lstm_test_pred,
}, index=idx_test)

ax = test_price_plot.plot(title=f'Тест: прогноз цены закрытия ETLN через {FORECAST_HORIZON} дней (RUB)')
ax.set_ylabel('RUB')
plt.show()

comparison.to_csv(OUTPUT_DIR / 'metrics_lstm_vs_gb_etln.csv', index=False)
test_price_plot.to_csv(OUTPUT_DIR / 'test_predictions_etln_price.csv')

# -----------------------------
# Наглядный график: текущая цена + прогноз на горизонт FORECAST_HORIZON
# -----------------------------
last_window = data['Close'].iloc[-WINDOW:].values.astype('float32')
last_close = float(data['Close'].iloc[-1])
last_date = data.index[-1]
forecast_date = last_date + pd.offsets.BDay(FORECAST_HORIZON)

# Масштабируем окно теми же параметрами train
last_window_scaled = (last_window - x_mean) / x_std

# Прогнозы (в scaled), затем возвращаем в рубли
forecast_gb_scaled = float(gb_model.predict(last_window_scaled.reshape(1, -1))[0])
forecast_lstm_scaled = float(lstm_model.predict(last_window_scaled.reshape(1, WINDOW, 1), verbose=0).ravel()[0])

forecast_gb_price = forecast_gb_scaled * y_std + y_mean
forecast_lstm_price = forecast_lstm_scaled * y_std + y_mean

# Берем последние точки истории для графика
history_points = 60
history_plot = data['Close'].tail(history_points)

ax = history_plot.plot(
    color='tab:blue',
    linewidth=2,
    label='current price',
    title=f'Текущая цена ETLN + прогноз на {FORECAST_HORIZON} торговых дней',
)
ax.set_ylabel('RUB')

# Вертикальная линия отделяет историю от прогноза
ax.axvline(last_date, color='gray', linestyle='--', linewidth=1)

# Отрезки вправо от последней цены до прогнозов
ax.plot(
    [last_date, forecast_date],
    [last_close, forecast_gb_price],
    color='tab:orange', linewidth=3,
    label=f'GB forecast ({FORECAST_HORIZON}d)'
)
ax.plot(
    [last_date, forecast_date],
    [last_close, forecast_lstm_price],
    color='tab:green', linewidth=3,
    label=f'LSTM forecast ({FORECAST_HORIZON}d)'
)

# Точки на концах прогнозов
ax.scatter([forecast_date], [forecast_gb_price], color='tab:orange', s=60)
ax.scatter([forecast_date], [forecast_lstm_price], color='tab:green', s=60)

# Подписи значений
ax.annotate(f'GB: {forecast_gb_price:.2f}', (forecast_date, forecast_gb_price), xytext=(8, 8), textcoords='offset points', color='tab:orange')
ax.annotate(f'LSTM: {forecast_lstm_price:.2f}', (forecast_date, forecast_lstm_price), xytext=(8, -14), textcoords='offset points', color='tab:green')

ax.legend()
plt.show()

print('Последняя известная цена:', round(last_close, 4), 'RUB на', last_date.date())
print(f'Прогноз GB на {FORECAST_HORIZON} дней  :', round(float(forecast_gb_price), 4), 'RUB')
print(f'Прогноз LSTM на {FORECAST_HORIZON} дней:', round(float(forecast_lstm_price), 4), 'RUB')
import json
latest_signal_payload = {
    'figi': FIGI,
    'forecast_horizon': int(FORECAST_HORIZON),
    'signal_date': str(pd.Timestamp(last_date).date()),
    'forecast_target_date': str(pd.Timestamp(forecast_date).date()),
    'current_price': float(last_close),
    'gb_horizon_price': float(forecast_gb_price),
    'lstm_horizon_price': float(forecast_lstm_price),
}
latest_signal_path = OUTPUT_DIR / f'latest_forecast_signal_etln_h{FORECAST_HORIZON}.json'
latest_signal_path.write_text(json.dumps(latest_signal_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Файл сигнала сохранен в:', latest_signal_path.resolve())

print('Файлы сохранены в:', OUTPUT_DIR.resolve())


## 11. Итоги для студентов

Что важно запомнить:
- Для временных рядов делаем **разбиение по времени**, а не случайное.
- Для честного сравнения масштабируем данные по статистикам **train** и применяем одинаково к обеим моделям.
- `Gradient Boosting` часто дает сильный базовый ориентир даже без нейросети.
- `LSTM` требует 3D-входа: `(samples, timesteps, features)`.
- Прогноз на `10` дней вперед обычно сложнее, чем на `1` день.

Идеи для самостоятельной работы:
- добавить наивный базовый ориентир: `прогноз = последняя цена в окне`;
- перейти к `log_close` или `log_return` и сравнить результаты;
- поменять `WINDOW` (например, 10 / 30 / 60);
- сравнить горизонты `1`, `5`, `10` дней.
